# 03 — Modeling & Results

**Research question:** Do teams that reached the 2022 World Cup quarterfinals show structurally different passing networks compared to teams eliminated earlier?

**Analytical pipeline:**
1. Statistical comparison (Mann-Whitney U, effect sizes) — is there a signal?
2. Classification (Logistic Regression + Random Forest, Leave-One-Out CV) — how predictive is it?
3. Feature importance — which network metrics matter most?
4. Team advancement profiles (K-Means + PCA) — what "types" of networks exist?
5. Radar chart comparison of selected teams

**Important framing:** With n=32 teams this is an exploratory analysis. The goal is to identify "advancement profiles," not to claim we can predict tournament results. LOO cross-validation is used because it is the correct strategy for small datasets.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from modeling import (
    compare_groups, descriptive_stats, evaluate_classifiers,
    fit_and_report, cluster_teams, profile_clusters, FEATURE_SETS
)
from visualization import (
    plot_feature_importance, plot_cluster_scatter, plot_team_radar
)

PROC_DIR = Path('../data/processed')
FIG_DIR  = Path('../outputs/figures')
TAB_DIR  = Path('../outputs/tables')

for d in [FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
match_features = pd.read_csv(PROC_DIR / 'match_features.csv')
team_features  = pd.read_csv(PROC_DIR / 'team_features.csv')

print(f"Match-level rows: {len(match_features)} | Team-level rows: {len(team_features)}")
print(f"QF teams: {team_features['reached_quarterfinal'].sum()} / {len(team_features)}")

## 1. Descriptive statistics by advancement group

In [ ]:
MATCH_FEATS = [
    'network_density', 'degree_centralization', 'top_player_pass_share',
    'clustering_coefficient', 'mean_betweenness_centrality',
    'progressive_pass_share', 'final_third_entry_share', 'lateral_imbalance',
    'centre_channel_share',
]
MATCH_FEATS = [f for f in MATCH_FEATS if f in match_features.columns]

desc = descriptive_stats(match_features, MATCH_FEATS)
desc_pivot = desc.pivot(index='feature', columns='group', values=['mean','std','median'])
desc_pivot.columns = [f"{stat}_{grp}" for stat, grp in desc_pivot.columns]
print(desc_pivot.round(4).to_string())

## 2. Statistical testing — Mann-Whitney U

In [ ]:
stat_results = compare_groups(
    match_features, MATCH_FEATS, group_col='reached_quarterfinal'
)

# Save table
stat_results.to_csv(TAB_DIR / 'stat_tests_match_level.csv', index=False)

print("Mann-Whitney U results (match-level features):")
stat_results[['feature','mean_qf','mean_elim','difference',
               'effect_size_rbc','p_value','significant']].round(4)

In [ ]:
# Effect size plot
sig   = stat_results[stat_results['significant']]
insig = stat_results[~stat_results['significant']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(sig['feature'], sig['effect_size_rbc'],
        color='#1f77b4', alpha=0.8, label='p < 0.05')
ax.barh(insig['feature'], insig['effect_size_rbc'],
        color='#aec7e8', alpha=0.6, label='p ≥ 0.05')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Effect size (rank-biserial correlation, positive = higher in QF teams)')
ax.set_title('Statistical Differences: QF+ vs Eliminated Teams (Match Level)')
ax.legend()
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.savefig(FIG_DIR / '03_effect_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Classification — team-level features

In [ ]:
# Evaluate all three feature sets with LOO CV
cv_results = []
for fset in ['network_only', 'pass_quality', 'combined']:
    res = evaluate_classifiers(
        team_features, feature_set=fset,
        target='reached_quarterfinal', cv_strategy='loo'
    )
    cv_results.append(res)

cv_df = pd.concat(cv_results, ignore_index=True)
cv_df.to_csv(TAB_DIR / 'classifier_cv_results.csv', index=False)
cv_df.sort_values('roc_auc_mean', ascending=False).round(3)

In [ ]:
# Bar chart of AUC by model × feature set
fig, ax = plt.subplots(figsize=(9, 4))
x     = np.arange(len(cv_df))
bars  = ax.bar(x, cv_df['roc_auc_mean'], color='#1f77b4', alpha=0.75)
ax.errorbar(x, cv_df['roc_auc_mean'], yerr=cv_df['roc_auc_std'],
            fmt='none', color='black', capsize=4)
ax.axhline(0.5, color='red', linestyle='--', lw=1, label='Random baseline (0.50)')
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['model'].replace('_',' ')}\n({r['feature_set']})" for _, r in cv_df.iterrows()],
    fontsize=8
)
ax.set_ylabel('ROC-AUC (LOO-CV)')
ax.set_title('Classifier Performance: Predicting QF Advancement from Network Features')
ax.set_ylim(0, 1)
ax.legend()
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.savefig(FIG_DIR / '03_classifier_auc.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature importance (Random Forest)

In [ ]:
pipeline, importances, predictions = fit_and_report(
    team_features, feature_set='combined', target='reached_quarterfinal'
)

importances.to_csv(TAB_DIR / 'feature_importances.csv', index=False)
print(importances.head(10).to_string(index=False))

In [ ]:
fig = plot_feature_importance(
    importances,
    top_n=9,
    save_path=FIG_DIR / '03_feature_importances.png',
)
plt.show()

## 5. Per-team predictions

In [ ]:
predictions = predictions.merge(
    team_features[['team','matches_played']], on='team', how='left'
).sort_values('predicted_proba', ascending=False)

predictions.to_csv(TAB_DIR / 'team_predictions.csv', index=False)
predictions[['team','reached_quarterfinal','predicted_proba','predicted_label']].round(3)

In [ ]:
# Who did the model rank highly but was eliminated (and vice versa)?
fp = predictions[(predictions['reached_quarterfinal']==0) & (predictions['predicted_label']==1)]
fn = predictions[(predictions['reached_quarterfinal']==1) & (predictions['predicted_label']==0)]

print("False positives (eliminated but network said QF):")
print(fp[['team','predicted_proba']].to_string(index=False))

print("\nFalse negatives (reached QF but network said eliminated):")
print(fn[['team','predicted_proba']].to_string(index=False))

## 6. Advancement profiles — K-Means clustering

In [ ]:
team_clustered = cluster_teams(team_features, feature_set='combined', n_clusters=3)

profiles = profile_clusters(team_clustered, feature_set='combined')
print("Cluster profiles (mean features):")
profiles.round(3)

In [ ]:
fig = plot_cluster_scatter(
    team_clustered,
    group_col='reached_quarterfinal',
    label_teams=True,
    save_path=FIG_DIR / '03_cluster_scatter.png',
)
plt.show()

In [ ]:
# QF advancement rate by cluster
cluster_summary = (
    team_clustered
    .groupby('cluster')
    .agg(
        n_teams=('team','count'),
        qf_teams=('reached_quarterfinal','sum'),
        qf_rate=('reached_quarterfinal','mean'),
    )
    .reset_index()
)
print("Cluster advancement rates:")
cluster_summary

## 7. Radar chart — compare selected teams

In [ ]:
RADAR_FEATS = [
    f for f in [
        'network_density_mean', 'degree_centralization_mean',
        'top_player_pass_share_mean', 'progressive_pass_share_mean',
        'final_third_entry_share_mean', 'centre_channel_share_mean',
    ]
    if f in team_features.columns
]

# Compare champions vs one eliminated team
available = team_features['team'].tolist()
radar_teams = [t for t in ['Argentina', 'France', 'Morocco', 'Australia'] if t in available][:4]

fig = plot_team_radar(
    team_features,
    team_names=radar_teams,
    features=RADAR_FEATS,
    save_path=FIG_DIR / '03_team_radar.png',
)
plt.show()

## 8. Results summary

In [ ]:
best_auc = cv_df.loc[cv_df['roc_auc_mean'].idxmax()]
sig_feats = stat_results[stat_results['significant']]['feature'].tolist()
top_feat  = importances.iloc[0]['feature']

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"\n1. Statistical testing")
print(f"   Significant features (p<0.05): {sig_feats if sig_feats else 'none — all trends, no significance'}")

print(f"\n2. Best classifier: {best_auc['model']} ({best_auc['feature_set']})")
print(f"   LOO-CV ROC-AUC: {best_auc['roc_auc_mean']:.3f} ± {best_auc['roc_auc_std']:.3f}")

print(f"\n3. Top predictive feature: {top_feat}")

print(f"\n4. Cluster QF rates:")
for _, r in cluster_summary.iterrows():
    print(f"   Cluster {int(r['cluster'])}: {int(r['n_teams'])} teams, "
          f"{int(r['qf_teams'])} QF ({r['qf_rate']:.0%})")

print("\n5. Interpretation:")
print("   Network structure alone does not perfectly predict World Cup advancement.")
print("   However, passing network metrics — particularly distribution of passing")
print("   load, network density, and progressive pass share — are associated with")
print("   teams that advance further, and clusters reveal distinct tactical profiles.")

## 9. Limitations and future work

**Limitations:**
- n=32 teams is small; statistical power is limited
- Network structure varies within a team across matches (injury, opponent, game state)
- Result (scoreline) influences late-game passing patterns, confounding tactical structure
- GK involvement feature is missing without lineup position integration

**Extensions:**
1. Integrate lineup position data to refine GK involvement and positional role features
2. Time-window analysis: networks in 0–45 vs. 46–90 min, or before/after goals
3. Control for opponent passing network (differential features)
4. Apply to WC 2026 data once StatsBomb releases it
5. Add xG and shot quality features as outcome metrics alongside advancement